In [ ]:
from typing import TypedDict,Literal

from jedi.inference.value.iterable import Sequence
from langchain_core.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek
from dotenv import load_dotenv
from langgraph.constants import START, END
from langgraph.graph import StateGraph

load_dotenv(override=True)

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={
        "thinking": {
            "type": "disabled"
        }
    }
)

# 1. 定义状态
class OverAllState(TypedDict):
    topic: str
    poem: str
    joke: str
    ci_poem: str
    content_type:str

# 2. 定义节点
def node_a(state: OverAllState) -> OverAllState:
    response = model.invoke([HumanMessage(content=f"写一首关于{state['topic']}的唐诗")])
    return {"poem": response.content}

def node_b(state: OverAllState) -> OverAllState:
    response = model.invoke([HumanMessage(content=f"写一个关于{state['topic']}的笑话")])
    return {"joke": response.content}
def node_c(state: OverAllState) -> OverAllState:
    response = model.invoke([HumanMessage(content=f"写一首关于{state['topic']}的唐诗")])
    return {"ci_poem": response.content}
def route(state: OverAllState) ->Sequence[Literal["a", "b", "c"]]:
    if "诗" in state["content_type"]:
        return ["a", "c"]
    else:
        return ["b"]

# 3. 构建图
builder = StateGraph(state_schema=OverAllState)
builder.add_node("node_a", node_a)
builder.add_node("node_b", node_b)

builder.add_conditional_edges(START,route,path_map={
    "a": "node_a",
    "b": "node_b"
})
builder.add_edge("node_a", END)
builder.add_edge("node_b", END)

graph = builder.compile()
graph.invoke({"topic": "猫猫", "content_type": "诗"})